## **Boosting**

#### **Regression**

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import xgboost as xgb

df = pd.read_csv('china_used_cars.csv')
drop_cols = ['price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['price']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# AdaBoost — boosts shallow trees via sample re-weighting
ada = AdaBoostRegressor(estimator=DecisionTreeRegressor(max_depth=4, random_state=42),
                         n_estimators=200, learning_rate=0.5, random_state=42)
ada.fit(X_train, y_train)
p_ada = ada.predict(X_test)
print("AdaBoost         -> MAE:", mean_absolute_error(y_test, p_ada), " R2:", r2_score(y_test, p_ada))

# Gradient Boosting — fits each tree to the residual error
gb = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
p_gb = gb.predict(X_test)
print("GradientBoosting -> MAE:", mean_absolute_error(y_test, p_gb), " R2:", r2_score(y_test, p_gb))

# XGBoost — regularized, optimized gradient boosting
xgbr = xgb.XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1)
xgbr.fit(X_train, y_train)
p_xgb = xgbr.predict(X_test)
print("XGBoost          -> MAE:", mean_absolute_error(y_test, p_xgb), " R2:", r2_score(y_test, p_xgb))

AdaBoost         -> MAE: 40257.9984528697  R2: 0.5644809542909655
GradientBoosting -> MAE: 18772.032623292354  R2: 0.44491038966434704
XGBoost          -> MAE: 15825.595924021949  R2: 0.5393802634496715


In [2]:
for lr, n in [(0.5, 50), (0.1, 100), (0.05, 200), (0.01, 500)]:
    g = GradientBoostingRegressor(n_estimators=n, learning_rate=lr, max_depth=3, random_state=42)
    g.fit(X_train, y_train)
    p = g.predict(X_test)
    print(f"lr={lr}  n_estimators={n} -> MAE: {mean_absolute_error(y_test,p):.1f}  R2: {r2_score(y_test,p):.4f}")

lr=0.5  n_estimators=50 -> MAE: 20532.0  R2: 0.1834
lr=0.1  n_estimators=100 -> MAE: 19420.7  R2: 0.3796
lr=0.05  n_estimators=200 -> MAE: 18772.0  R2: 0.4449
lr=0.01  n_estimators=500 -> MAE: 18946.0  R2: 0.4668


In [3]:
g_over = GradientBoostingRegressor(n_estimators=1000, learning_rate=0.2, max_depth=5, random_state=42)
g_over.fit(X_train, y_train)
print("Train R2:", r2_score(y_train, g_over.predict(X_train)))
print("Test  R2:", r2_score(y_test, g_over.predict(X_test)))

Train R2: 0.9995458947597737
Test  R2: 0.46384020761711


#### **Classification**

In [4]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import xgboost as xgb

drop_cols = ['is_electric', 'battery_capacity_kwh', 'motor_power_kw',
             'price', 'mileage_km', 'log_mileage', 'mileage_per_year', 'year', 'month']
X = df.drop(columns=drop_cols)
y = df['is_electric']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2, random_state=42),
                          n_estimators=200, random_state=42)
ada.fit(X_train, y_train)
print("AdaBoost acc:", accuracy_score(y_test, ada.predict(X_test)))

gb = GradientBoostingClassifier(n_estimators=200, learning_rate=0.05, max_depth=3, random_state=42)
gb.fit(X_train, y_train)
print("GradientBoosting acc:", accuracy_score(y_test, gb.predict(X_test)))

xgbc = xgb.XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, random_state=42, n_jobs=-1, eval_metric='logloss')
xgbc.fit(X_train, y_train)
print("XGBoost acc:", accuracy_score(y_test, xgbc.predict(X_test)))

AdaBoost acc: 0.989247311827957
GradientBoosting acc: 0.9904420549581839
XGBoost acc: 0.98805256869773
